# SBOLInventory examples

This notebook demonstrates how to use **SBOLInventory** to build a storage hierarchy, create inventory implementations, place them into slots, validate placements, and assemble an SBOL document.

In [ ]:
import sbol2 as sbol

from sbol_inventory import (
    make_document,
    add_all,
    make_fridge_minus80,
    make_fridge_minus20,
    make_fridge_4c,
    make_shelf,
    make_box,
    make_slot,
    make_bacterial_stock,
    make_extracted_plasmid,
    make_solid_media_plate,
    add_child,
    place_item,
    validate_item,
    validate_placement,
)

## Example 1: create a -80 °C hierarchy and place a bacterial stock

In [ ]:
doc = make_document()

freezer = make_fridge_minus80("https://example.org/storage/-80")
shelf = make_shelf("https://example.org/storage/-80/shelf2", label="Shelf 2")
box = make_box("https://example.org/storage/-80/shelf2/box4", label="Box 4")
slot = make_slot("https://example.org/storage/-80/shelf2/box4/A4", label="A4")

stock = make_bacterial_stock(
    uri="https://example.org/implementation/bstock_001",
    strain_md_uri="https://example.org/designs/strain_md_001",
)

add_all(doc, [freezer, shelf, box, slot, stock])
add_child(freezer, shelf)
add_child(shelf, box)
add_child(box, slot)
place_item(slot, stock)

validate_item(stock)
validate_placement(stock, slot)

print("Stored at:", stock.stored_at)
print("Slot members:", list(slot.members))

## Example 2: create a -20 °C hierarchy and place an extracted plasmid

In [ ]:
freezer20 = make_fridge_minus20("https://example.org/storage/-20")
shelf20 = make_shelf("https://example.org/storage/-20/shelf1", label="Shelf 1")
box20 = make_box("https://example.org/storage/-20/shelf1/box2", label="Box 2")
slot20 = make_slot(
    "https://example.org/storage/-20/shelf1/box2/B3",
    label="B3",
    allowed_item_kinds=["https://draggon.org/ns/inventory#ExtractedPlasmid"],
)

plasmid = make_extracted_plasmid(
    uri="https://example.org/implementation/plasmid_001",
    plasmid_cd_uri="https://example.org/designs/plasmid_cd_001",
)

for obj in [freezer20, shelf20, box20, slot20, plasmid]:
    doc.add(obj)

add_child(freezer20, shelf20)
add_child(shelf20, box20)
add_child(box20, slot20)
place_item(slot20, plasmid)

validate_item(plasmid)
validate_placement(plasmid, slot20)

print("Plasmid stored at:", plasmid.stored_at)

## Example 3: create a 4 °C hierarchy and place a solid media plate

In [ ]:
fridge4 = make_fridge_4c("https://example.org/storage/4c")
shelf4 = make_shelf("https://example.org/storage/4c/shelf1", label="Shelf 1")
box4 = make_box("https://example.org/storage/4c/shelf1/rack1", label="Rack 1")
slot4 = make_slot(
    "https://example.org/storage/4c/shelf1/rack1/P1",
    label="P1",
    allowed_item_kinds=["https://draggon.org/ns/inventory#SolidMediaPlate"],
)

plate = make_solid_media_plate(
    uri="https://example.org/implementation/plate_001",
    plate_md_uri="https://example.org/designs/plate_md_001",
)

for obj in [fridge4, shelf4, box4, slot4, plate]:
    doc.add(obj)

add_child(fridge4, shelf4)
add_child(shelf4, box4)
add_child(box4, slot4)
place_item(slot4, plate)

validate_item(plate)
validate_placement(plate, slot4)

print("Plate stored at:", plate.stored_at)

## Example 4: show a validation failure

In [ ]:
wrong_slot = make_slot(
    "https://example.org/storage/-80/shelf9/box9/Z9",
    label="Z9",
    allowed_item_kinds=["https://draggon.org/ns/inventory#BacterialStock"],
)

wrong_item = make_extracted_plasmid(
    uri="https://example.org/implementation/plasmid_bad",
    plasmid_cd_uri="https://example.org/designs/plasmid_cd_bad",
)

try:
    validate_placement(wrong_item, wrong_slot)
except ValueError as e:
    print("Validation error:", e)

## Example 5: serialize the document

In [ ]:
# Uncomment to write RDF/XML once sbol2 is installed and configured:
# from sbol_inventory import write_rdfxml
# write_rdfxml(doc, "inventory_example.xml")
# print("Wrote inventory_example.xml")